In [1]:
import os, shutil
print('vCPUs:', os.cpu_count())
print('RAM GB:', round(int(open('/proc/meminfo').readline().split()[1]) / 1024**2, 1))
print('disk free GB:', round(shutil.disk_usage('/content').free / 1024**3, 1))

vCPUs: 48
RAM GB: 176.9
disk free GB: 188.9


# Othello JEPA Colab Training

This notebook controls EMA-target JEPA pretraining for the Othello Transformer encoder. The training logic lives in `othello_research.training.train_jepa`.

## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Use Drive-Synced Project Code

In [3]:
from pathlib import Path
import os
os.environ['OTHELLO_DISABLE_TQDM'] = '1'
import subprocess
import sys

PROJECT_ROOT = Path('/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/Master_Thesis_Artifacts')
DATA_ROOT = ARTIFACT_ROOT / 'data'
RUNS_DIR = ARTIFACT_ROOT / 'runs'
CONFIGS_DIR = PROJECT_ROOT / 'configs'

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert (PROJECT_ROOT / 'pyproject.toml').exists(), PROJECT_ROOT
assert (PROJECT_ROOT / 'othello_research').is_dir(), PROJECT_ROOT
assert ARTIFACT_ROOT.exists(), ARTIFACT_ROOT
assert DATA_ROOT.exists(), DATA_ROOT
assert RUNS_DIR.exists(), RUNS_DIR
assert CONFIGS_DIR.exists(), CONFIGS_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('ARTIFACT_ROOT:', ARTIFACT_ROOT)
print('DATA_ROOT    :', DATA_ROOT)
print('RUNS_DIR     :', RUNS_DIR)
print('CONFIGS_DIR  :', CONFIGS_DIR)

PROJECT_ROOT : /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code
ARTIFACT_ROOT: /content/drive/MyDrive/Master_Thesis_Artifacts
DATA_ROOT    : /content/drive/MyDrive/Master_Thesis_Artifacts/data
RUNS_DIR     : /content/drive/MyDrive/Master_Thesis_Artifacts/runs
CONFIGS_DIR  : /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code/configs


## 3. Configure paths

In [4]:
from pathlib import Path
import yaml

# === SINGLE SOURCE OF TRUTH ===
# Change this filename to launch a different JEPA experiment.
CONFIG_NAME = "jepa_v4_multi_pos_k4_hd_all_positions.yml"
CONFIG_PATH = CONFIGS_DIR / CONFIG_NAME
assert CONFIG_PATH.exists(), CONFIG_PATH

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    EXPERIMENT_CONFIG = yaml.safe_load(f)

def config_value(section, key, default=None):
    section_data = EXPERIMENT_CONFIG.get(section, {})
    if isinstance(section_data, dict) and key in section_data:
        return section_data[key]
    return EXPERIMENT_CONFIG.get(key, default)

BOARD_SIZE = int(config_value('data', 'board_size', 8))
PREDICTOR_TYPE = config_value('predictor', 'predictor_type')
LOSS_TYPE = config_value('objective', 'loss_type', 'auto')
PREDICTION_HORIZON = int(config_value('objective', 'prediction_horizon', 1))
OBJECTIVE_CLASS = str(config_value('objective', 'objective_class', 'jepa'))
RUN_NAME = str(config_value('experiment', 'name'))
N_TRAIN_CHUNKS = int(config_value('data', 'li_split_train_chunks', 200))
PAIR_INDEX_VALUE = config_value('paths', 'pair_index_path')
PAIR_INDEX_PATH = Path(PAIR_INDEX_VALUE) if PAIR_INDEX_VALUE else None

DATA_DIR = DATA_ROOT / f'othello_{BOARD_SIZE}x{BOARD_SIZE}'

# Train on local Colab disk for speed, then sync checkpoints/results to MyDrive artifacts.
LOCAL_OUT_DIR = Path('/content/runs') / RUN_NAME
DRIVE_OUT_DIR = RUNS_DIR / RUN_NAME
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

all_chunks = sorted(DATA_DIR.glob('*.pickle'))
train_chunks = all_chunks[:N_TRAIN_CHUNKS]
val_chunks = all_chunks[N_TRAIN_CHUNKS:]

print('CONFIG_NAME    :', CONFIG_NAME)
print('CONFIG_PATH    :', CONFIG_PATH)
print('RUN_NAME       :', RUN_NAME)
print('OBJECTIVE_CLASS:', OBJECTIVE_CLASS)
print('BOARD_SIZE     :', BOARD_SIZE)
print('PREDICTOR_TYPE :', PREDICTOR_TYPE)
print('LOSS_TYPE      :', LOSS_TYPE)
print('K / horizon    :', PREDICTION_HORIZON)
print('DATA_DIR       :', DATA_DIR)
print('PAIR_INDEX_PATH:', PAIR_INDEX_PATH)
print('LOCAL_OUT_DIR  :', LOCAL_OUT_DIR)
print('DRIVE_OUT_DIR  :', DRIVE_OUT_DIR)
print('train chunks   :', len(train_chunks))
print('val chunks     :', len(val_chunks))

assert DATA_DIR.exists(), DATA_DIR
assert train_chunks, f'No training chunks found in {DATA_DIR}'
assert val_chunks, f'No validation chunks found after the first {N_TRAIN_CHUNKS} chunks'

CONFIG_NAME    : jepa_v4_multi_pos_k4_hd_all_positions.yml
CONFIG_PATH    : /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code/configs/jepa_v4_multi_pos_k4_hd_all_positions.yml
RUN_NAME       : jepa_v4_multi_pos_k4_hd_run_001_allpos_bf16
OBJECTIVE_CLASS: jepa
BOARD_SIZE     : 8
PREDICTOR_TYPE : mlp_multi_pos
LOSS_TYPE      : smooth_l1
K / horizon    : 4
DATA_DIR       : /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_8x8
PAIR_INDEX_PATH: None
LOCAL_OUT_DIR  : /content/runs/jepa_v4_multi_pos_k4_hd_run_001_allpos_bf16
DRIVE_OUT_DIR  : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/jepa_v4_multi_pos_k4_hd_run_001_allpos_bf16
train chunks   : 200
val chunks     : 38


### v8 offline pair index

For `jepa_order_aware`, this cell validates the cached pair index and builds it once when missing. Training batches never invoke the Othello engine.


In [5]:
!pip -q install duckdb

In [6]:
import codecs
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from pathlib import Path

from othello_research.datasets.order_aware_pair_index import INDEX_VERSION


V8_OBJECTIVE_NAMES = {'jepa_order_aware', 'order_aware', 'v8'}
FAMILY_OBJECTIVE_NAMES = {'family_infonce', 'family', 'transposition_family'}


def expected_v8_index_settings():
    expected_constructive = bool(
        config_value('pair_index', 'order_index_use_constructive_pairs', False)
    )
    expected_surface_hard_negatives = bool(
        config_value('pair_index', 'order_index_use_surface_hard_negatives', False)
    )
    expected_surface_jaccard_threshold = float(
        config_value('pair_index', 'order_surface_jaccard_threshold', 0.9)
    )
    expected_surface_max_pairs = int(
        config_value('pair_index', 'order_surface_max_pairs_per_anchor', 1)
    )
    return {
        'version': INDEX_VERSION,
        'board_size': BOARD_SIZE,
        't_min': int(config_value('pair_index', 'order_t_min', 4)),
        't_max': int(config_value('pair_index', 'order_t_max', 40)),
        'positions_per_game': int(
            config_value('pair_index', 'order_index_positions_per_game', 1)
        ),
        'sampling_seed': int(
            config_value('pair_index', 'order_index_sampling_seed', 42)
        ),
        'use_constructive_pairs': expected_constructive,
        'use_surface_hard_negatives': expected_surface_hard_negatives,
        'surface_jaccard_threshold': expected_surface_jaccard_threshold,
        'surface_max_pairs_per_anchor': expected_surface_max_pairs,
        'n_train_chunks': N_TRAIN_CHUNKS,
    }


def read_v8_index_stats(path):
    """Return coverage stats only when the SQLite metadata matches this YAML."""
    try:
        connection = sqlite3.connect(str(path))
        try:
            metadata = {
                key: json.loads(value)
                for key, value in connection.execute('SELECT key, value FROM metadata')
            }
        finally:
            connection.close()
        expected = expected_v8_index_settings()
        coverage = metadata.get('coverage_stats') or {}
        if (
            int(metadata.get('version', -1)) != expected['version']
            or int(metadata.get('board_size', -1)) != expected['board_size']
            or int(metadata.get('t_min', expected['t_min'] + 1)) > expected['t_min']
            or int(metadata.get('t_max', expected['t_max'] - 1)) < expected['t_max']
            or int(metadata.get('positions_per_game', -1)) != expected['positions_per_game']
            or int(metadata.get('sampling_seed', -1)) != expected['sampling_seed']
            or bool(metadata.get('use_constructive_pairs', False))
            != expected['use_constructive_pairs']
            or bool(metadata.get('use_surface_hard_negatives', False))
            != expected['use_surface_hard_negatives']
            or abs(
                float(metadata.get('surface_jaccard_threshold', 0.9))
                - expected['surface_jaccard_threshold']
            )
            > 1e-9
            or int(metadata.get('surface_max_pairs_per_anchor', 1))
            != expected['surface_max_pairs_per_anchor']
            or int(coverage.get('n_chunks', -1)) != expected['n_train_chunks']
        ):
            return None
        return coverage
    except Exception:
        return None


def run_build_streaming(command):
    """Stream every subprocess line straight to the cell output."""
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    try:
        _decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
        while True:
            _data = process.stdout.read1(8192)
            if not _data:
                break
            sys.stdout.write(_decoder.decode(_data))
            sys.stdout.flush()
        _tail = _decoder.decode(b'', True)
        if _tail:
            sys.stdout.write(_tail)
            sys.stdout.flush()
    finally:
        process.stdout.close()
        rc = process.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, command)


def build_v8_pair_index_command(output_path, build_workers):
    expected = expected_v8_index_settings()
    command = [
        sys.executable,
        '-u',  # unbuffered stdout so the per-chunk progress lines stream live
        str(PROJECT_ROOT / 'tools' / 'build_v8_pair_index.py'),
        '--data_dir', str(DATA_DIR),
        '--output', str(output_path),
        '--board_size', str(expected['board_size']),
        '--train_chunks', str(expected['n_train_chunks']),
        '--t_min', str(expected['t_min']),
        '--t_max', str(expected['t_max']),
        '--positions_per_game', str(expected['positions_per_game']),
        '--sampling_seed', str(expected['sampling_seed']),
        '--surface_jaccard_threshold', str(expected['surface_jaccard_threshold']),
        '--surface_max_pairs_per_anchor', str(expected['surface_max_pairs_per_anchor']),
        '--workers', str(build_workers),
    ]
    if expected['use_constructive_pairs']:
        command.append('--use_constructive_pairs')
    if expected['use_surface_hard_negatives']:
        command.append('--use_surface_hard_negatives')
    if Path(output_path).exists():
        command.append('--overwrite')
    return command




def ensure_family_artifacts():
    """Build/cache/stage the transposition-family artifacts (family_infonce).

    Build once on a CPU-heavy runtime (the expensive scan+aggregate), cache on
    Drive, and copy Drive -> local NVMe on every later runtime. Returns the
    runtime args the training cells forward to train_jepa.py.
    """
    drive_dir = Path(config_value('paths', 'artifacts_dir'))
    local_dir = Path(config_value('paths', 'local_stage_dir',
                                  '/content/family_artifacts'))
    t_min = int(config_value('family_index', 'family_t_min', 4))
    t_max = int(config_value('family_index', 'family_t_max', 44))
    # Scan work is embarrassingly parallel per chunk (~200-300 MB RAM per
    # worker), so default to every core but one. Set family_build_workers in
    # the YAML only to throttle (e.g. thermals on a laptop).
    default_workers = max(1, (os.cpu_count() or 2) - 1)
    workers = int(config_value('family_index', 'family_build_workers', 0)
                  or default_workers)

    def copy_tree(src, dst):
        copied = 0
        for path in sorted(Path(src).rglob('*')):
            if path.is_dir() or '_intermediate' in path.parts:
                continue
            rel = path.relative_to(src)
            target = Path(dst) / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists() or target.stat().st_size != path.stat().st_size:
                shutil.copy2(path, target)
                copied += 1
        return copied

    if (local_dir / 'manifest.json').exists():
        print('family artifacts already staged locally:', local_dir)
    elif (drive_dir / 'manifest.json').exists():
        print('Copying cached family artifacts from Drive to local disk...')
        print(f'  copied {copy_tree(drive_dir, local_dir)} file(s)')
    else:
        # One-off build (run this branch on a CPU-heavy runtime).
        # Colab preinstalls numba/pyarrow but NOT duckdb; install what is
        # missing into the same interpreter the build subprocess will use.
        for module_name in ('numba', 'duckdb', 'pyarrow'):
            try:
                __import__(module_name)
            except ImportError:
                print(f'Installing {module_name} (family build dependency)...',
                      flush=True)
                subprocess.run(
                    [sys.executable, '-m', 'pip', 'install', '-q', module_name],
                    check=True,
                )
        local_chunks = Path('/content/family_chunks')
        local_chunks.mkdir(parents=True, exist_ok=True)
        print(f'Staging {len(all_chunks)} chunks locally for the family scan...',
              flush=True)
        for i, src in enumerate(all_chunks, start=1):
            dst = local_chunks / src.name
            if not dst.exists() or dst.stat().st_size != src.stat().st_size:
                shutil.copy2(src, dst)
            if i % 25 == 0 or i == len(all_chunks):
                print(f'  staged {i}/{len(all_chunks)}', flush=True)
        run_build_streaming([
            sys.executable, '-u',
            str(PROJECT_ROOT / 'scripts' / 'build_family_index.py'),
            '--data_dir', str(local_chunks),
            '--out_dir', str(local_dir),
            '--train_chunks', str(N_TRAIN_CHUNKS),
            '--t_min', str(t_min), '--t_max', str(t_max),
            '--workers', str(workers),
            '--duckdb_memory_gb', '0',  # 0 = auto (~50% of RAM)
        ])
        print('Caching family artifacts to Drive...')
        print(f'  uploaded {copy_tree(local_dir, drive_dir)} file(s) '
              f'-> {drive_dir}')

    density_md = local_dir / 'density_report.md'
    if density_md.exists():
        print()
        print(density_md.read_text(encoding='utf-8'))
        print()
        print('STOP GATE: review the density report above before training.')
    print('family artifacts (local):', local_dir)
    return ['--artifacts_dir', str(local_dir)]


def ensure_v8_pair_index(force_rebuild=False):
    """Validate or build the v8 SQLite index and set PAIR_INDEX_RUNTIME_ARGS.

    Safe for all configs: non-v8 objectives just skip this setup.
    """
    global DATA_DIR, train_chunks, val_chunks
    global PAIR_INDEX_RUNTIME_PATH, PAIR_INDEX_RUNTIME_ARGS

    PAIR_INDEX_RUNTIME_ARGS = []
    if OBJECTIVE_CLASS in FAMILY_OBJECTIVE_NAMES:
        PAIR_INDEX_RUNTIME_PATH = None
        PAIR_INDEX_RUNTIME_ARGS = ensure_family_artifacts()
        return None
    if OBJECTIVE_CLASS not in V8_OBJECTIVE_NAMES:
        PAIR_INDEX_RUNTIME_PATH = PAIR_INDEX_PATH
        print('Pair index is not required for objective:', OBJECTIVE_CLASS)
        return None

    if PAIR_INDEX_PATH is None:
        raise ValueError('v8 requires paths.pair_index_path in the YAML config')

    # Pair prefixes are resolved from SQLite, so training reads each current
    # chunk from Drive on demand instead of copying the full dataset locally.
    DATA_DIR = DATA_ROOT / f'othello_{BOARD_SIZE}x{BOARD_SIZE}'
    all_chunks = sorted(DATA_DIR.glob('*.pickle'))
    train_chunks = all_chunks[:N_TRAIN_CHUNKS]
    val_chunks = all_chunks[N_TRAIN_CHUNKS:]
    print('v8 chunks will be loaded on demand from:', DATA_DIR, flush=True)

    LOCAL_PAIR_INDEX_DIR = Path('/content/v8_pair_indexes')
    LOCAL_PAIR_INDEX_DIR.mkdir(parents=True, exist_ok=True)
    PAIR_INDEX_RUNTIME_PATH = LOCAL_PAIR_INDEX_DIR / PAIR_INDEX_PATH.name

    expected = expected_v8_index_settings()
    print('v8 expected pair-index settings:')
    print(json.dumps(expected, indent=2))

    pair_stats = None
    if not force_rebuild and PAIR_INDEX_RUNTIME_PATH.exists():
        pair_stats = read_v8_index_stats(PAIR_INDEX_RUNTIME_PATH)
        if pair_stats is None:
            print('Local v8 pair index metadata mismatch; rebuilding local cache.')

    if not force_rebuild and pair_stats is None and PAIR_INDEX_PATH.exists():
        drive_stats = read_v8_index_stats(PAIR_INDEX_PATH)
        if drive_stats is not None:
            print('Copying matching v8 pair index from Drive to local disk...', flush=True)
            shutil.copy2(PAIR_INDEX_PATH, PAIR_INDEX_RUNTIME_PATH)
            pair_stats = read_v8_index_stats(PAIR_INDEX_RUNTIME_PATH)
        else:
            print('Drive v8 pair index metadata mismatch; it will be overwritten.')

    if force_rebuild or pair_stats is None:
        default_workers = max(1, min(8, (os.cpu_count() or 2)))
        build_workers = int(
            config_value('pair_index', 'pair_index_build_workers', default_workers)
        )
        command = build_v8_pair_index_command(PAIR_INDEX_RUNTIME_PATH, build_workers)
        print('Building the efficient v8 pair index on local disk:', flush=True)
        print(f'  workers={build_workers}, chunks={N_TRAIN_CHUNKS}', flush=True)
        print(' '.join(command), flush=True)
        print(
            f'(no progress lines yet means the {build_workers} workers are still '
            f'replaying their first chunk; expect the first [v8 build] line '
            f'within a few minutes)',
            flush=True,
        )
        run_build_streaming(command)
        pair_stats = read_v8_index_stats(PAIR_INDEX_RUNTIME_PATH)
        if pair_stats is None:
            raise RuntimeError(
                f'Built pair index metadata does not match this config: '
                f'{PAIR_INDEX_RUNTIME_PATH}'
            )
        PAIR_INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
        print('Caching completed v8 pair index to Drive...', flush=True)
        shutil.copy2(PAIR_INDEX_RUNTIME_PATH, PAIR_INDEX_PATH)

    PAIR_INDEX_RUNTIME_ARGS = ['--pair_index_path', str(PAIR_INDEX_RUNTIME_PATH)]
    print('v8 runtime pair index:', PAIR_INDEX_RUNTIME_PATH)
    print('v8 pair-index coverage:')
    print(json.dumps(pair_stats, indent=2))
    return pair_stats


def require_v8_pair_index_ready():
    """Lightweight training-cell guard; never builds or overwrites indexes."""
    if OBJECTIVE_CLASS not in V8_OBJECTIVE_NAMES:
        return
    if 'PAIR_INDEX_RUNTIME_PATH' not in globals() or PAIR_INDEX_RUNTIME_PATH is None:
        raise RuntimeError('Run the v8 offline pair-index cell before training.')
    if 'PAIR_INDEX_RUNTIME_ARGS' not in globals() or not PAIR_INDEX_RUNTIME_ARGS:
        raise RuntimeError('Run the v8 offline pair-index cell before training.')
    pair_stats = read_v8_index_stats(PAIR_INDEX_RUNTIME_PATH)
    if pair_stats is None:
        raise RuntimeError(
            'The runtime v8 pair index does not match this YAML config. '
            'Run the v8 offline pair-index cell before training.'
        )


pair_stats = ensure_v8_pair_index()

Pair index is not required for objective: jepa


In [7]:
import json, sqlite3
from pathlib import Path

PAIR_INDEX_PATH = Path('/content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_8x8/v8_pair_index.sqlite')
print('Drive index size:', round(PAIR_INDEX_PATH.stat().st_size / 1024**2, 1), 'MB')

con = sqlite3.connect(str(PAIR_INDEX_PATH))
metadata = {k: json.loads(v) for k, v in con.execute('SELECT key, value FROM metadata')}
coverage = metadata.get('coverage_stats', {})
con.close()

print('\nmetadata (non-stats):')
for k, v in metadata.items():
    if k == 'coverage_stats':
        continue
    print(f'  {k}: {v}')

print('\ncoverage stats:')
for k in [
    'n_chunks', 'n_games', 'n_occurrences', 'n_candidate_occurrences',
    'sampling_fraction', 'n_replayed_moves',
    'positive_count', 'hard_negative_count',
    'positive_anchor_count', 'hard_negative_anchor_count',
    'estimated_positive_fallback_rate', 'estimated_mean_hard_negatives_per_anchor',
]:
    if k in coverage:
        print(f'  {k}: {coverage[k]}')


Drive index size: 1738.4 MB

metadata (non-stats):
  version: 4
  board_size: 8
  t_min: 4
  t_max: 40
  use_constructive_pairs: True
  use_surface_hard_negatives: True
  surface_jaccard_threshold: 0.85
  surface_max_pairs_per_anchor: 2
  positions_per_game: 2
  sampling_seed: 42
  embedded_pair_prefixes: True

coverage stats:
  n_chunks: 50
  n_games: 4999952
  n_occurrences: 9999904
  n_candidate_occurrences: 184928702
  sampling_fraction: 0.05407437510700746
  n_replayed_moves: 156162490
  positive_count: 2354283
  hard_negative_count: 4119558
  positive_anchor_count: 2286574
  hard_negative_anchor_count: 3382050
  estimated_positive_fallback_rate: 0.7713404048678867
  estimated_mean_hard_negatives_per_anchor: 0.4119597548136462


## 4. Environment check

In [8]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print('vram GB:', round(props.total_memory / 1024**3, 2))

torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
vram GB: 94.97


## 5. Imports

In [9]:
import importlib
import math
import shutil
import torch

from othello_research.datasets.dataset import OthelloChunkDataset, load_chunk
from othello_research.datasets.move_mapping import build_mappings
import othello_research.objectives.jepa as jepa_obj
import othello_research.training.train_jepa as tjm

print('imports OK')
print('mapping sizes:', [len(x) for x in build_mappings(BOARD_SIZE)])

imports OK
mapping sizes: [64, 60]


## 6. Sanity check

In [10]:
import importlib
import math

import othello_research.objectives.jepa_action_conditioned as jac
import othello_research.objectives.jepa_order_aware as joa
import othello_research.objectives.jepa_order_aware as joa
import othello_research.objectives.jepa_hard_disjoint_action as jhda
import othello_research.objectives.jepa_hard_disjoint_infonce as jhdi
import othello_research.objectives.jepa as jepa_obj
import othello_research.datasets.order_aware_pair_index as order_pair_index
import othello_research.datasets.order_aware_dataset as order_dataset
import othello_research.training.train_jepa_order_aware as order_training
import othello_research.training.train_jepa as tjm

importlib.reload(jac)
importlib.reload(joa)
importlib.reload(joa)
importlib.reload(jhda)
importlib.reload(jhdi)
importlib.reload(jepa_obj)
importlib.reload(order_pair_index)
importlib.reload(order_dataset)
importlib.reload(order_training)
importlib.reload(tjm)

torch.manual_seed(42)

smoke_args, _, _ = tjm.parse_configured_args([
    '--config', str(CONFIG_PATH),
    '--out_dir', '/content/_jepa_sanity_model',
    '--data_dir', str(DATA_DIR),
    '--drive_sync_dir', '',
    '--n_layers', '2',
    '--n_heads', '4',
    '--d_model', '64',
    '--dropout', '0.1',
    '--predictor_hidden_dim', '128',
    '--predictor_hidden_mult', '2',
    '--predictor_n_layers', '2',
    '--predictor_n_heads', '4',
    '--predictor_dropout', '0.0',
    '--batch_size', '4',
    '--precision', 'fp32',
    '--probe_interval_steps', '0',
    '--order_groups_per_batch', '2',
    '--order_samples_per_group', '2',
    '--order_groups_per_batch', '2',
    '--order_samples_per_group', '2',
])
smoke_cfg, _ = tjm.namespace_to_train_config(smoke_args)
model, tiny_cfg = tjm.build_objective_model(smoke_cfg)
model = model.cuda() if torch.cuda.is_available() else model
device = next(model.parameters()).device
objective_name = tjm.objective_class(smoke_cfg)

if objective_name in {'jepa_hard_disjoint_action', 'jepa_hard_disjoint_infonce'}:
    batch_size = 8
    context_len = 4
    x_context = torch.randint(
        0, model.config.vocab_size, (batch_size, context_len), device=device
    )
    prefix_ids = torch.tensor(
        [0, 0, 1, 1, 2, 2, 3, 3], dtype=torch.long, device=device
    )
    for group_id in prefix_ids.unique():
        members = prefix_ids == group_id
        x_context[members] = x_context[members][0]

    vocab_actions = int(getattr(model, '_vocab_actions', model.config.vocab_size - 1))
    next_actions = torch.randint(0, vocab_actions, (batch_size,), device=device)
    next_actions[2] = next_actions[0]
    out = model(x_context, next_actions, prefix_ids)

elif objective_name == 'family_infonce':
    batch_size = 8
    t = 5
    vocab_actions = model.config.vocab_size - 1
    tokens = torch.randint(0, vocab_actions, (batch_size, t), device=device)
    lengths = torch.full((batch_size,), t, dtype=torch.long, device=device)
    family_ids = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], device=device)
    class_ids = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3], device=device)
    eligible = torch.ones(batch_size, dtype=torch.bool, device=device)
    out = model(tokens, lengths, family_ids, class_ids, eligible)

elif objective_name == 'jepa_order_aware':
    batch_size = 8
    context_len = 4
    vocab_actions = model.config.vocab_size - 1
    x_context = torch.randint(
        0, vocab_actions, (batch_size, context_len), device=device
    )
    group_ids = torch.arange(batch_size // 2, device=device).repeat_interleave(2)
    group_actions = torch.randint(
        0, vocab_actions, (batch_size // 2,), device=device
    )
    next_actions = group_actions.repeat_interleave(2)
    partner_context = x_context.reshape(batch_size // 2, 2, context_len).flip(1).reshape_as(x_context)
    x_positive_target = torch.cat([partner_context, next_actions.unsqueeze(1)], dim=1)
    x_hard_negative_targets = x_positive_target.unsqueeze(1).clone()
    x_hard_negative_targets[:, :, 0] = (x_hard_negative_targets[:, :, 0] + 1) % vocab_actions
    hard_negative_mask = torch.ones(batch_size, 1, dtype=torch.bool, device=device)
    positive_fallback_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    out = model(
        x_context,
        x_positive_target,
        x_hard_negative_targets,
        hard_negative_mask,
        group_ids,
        positive_fallback_mask,
    )

elif objective_name == 'jepa_action_conditioned':
    batch_size = 8
    context_len = 4
    vocab_actions = model.config.vocab_size - 1
    x_context = torch.randint(
        0, vocab_actions, (batch_size, context_len), device=device
    )
    uses_grouped_infonce = (
        smoke_cfg.action_loss_mode == 'grouped_infonce'
        or (
            smoke_cfg.action_loss_mode == 'hybrid'
            and smoke_cfg.hybrid_base == 'grouped_infonce'
        )
    )
    if uses_grouped_infonce:
        group_ids = torch.arange(batch_size // 2, device=device).repeat_interleave(2)
        group_actions = torch.randint(
            0, vocab_actions, (batch_size // 2,), device=device
        )
        next_actions = group_actions.repeat_interleave(2)
    else:
        group_ids = None
        next_actions = torch.randint(0, vocab_actions, (batch_size,), device=device)

    x_target = torch.cat([x_context, next_actions.unsqueeze(1)], dim=1)
    out = model(x_context, x_target, group_ids=group_ids)

else:
    K = tiny_cfg.prediction_horizon
    context_len = max(1, min(8, model.config.block_size - K))
    tokens = torch.randint(
        0, model.config.vocab_size, (4, context_len + K), device=device
    )
    x_context, x_target, target_positions = tjm.build_jepa_views(
        smoke_cfg,
        tokens,
        context_length=context_len,
        horizon=K,
    )
    out = model(x_context, x_target, target_positions=target_positions)

tjm.update_target_encoder(model)
print('objective_class:', smoke_cfg.objective_class)
print('resolved_objective_class:', objective_name)
print('view_mode:', getattr(tiny_cfg, 'view_mode', None))
print('predictor:', getattr(tiny_cfg, 'predictor_type', smoke_cfg.predictor_type))
if hasattr(tiny_cfg, 'prediction_horizon'):
    print('horizon:', tiny_cfg.prediction_horizon)
if hasattr(tiny_cfg, 'num_modes'):
    print('num_modes:', tiny_cfg.num_modes)
print('loss:', float(out['loss'].detach().cpu()))
for key in (
    'main_loss', 'base_loss', 'mse_loss', 'smooth_l1_loss',
    'inv_loss', 'var_loss', 'cov_loss', 'infonce_loss', 'hybrid_ce_loss',
    'action_only_loss', 'context_utility', 'shuffled_context_metric',
    'contrastive_accuracy', 'positive_accuracy', 'action_only_accuracy',
    'diag_accuracy', 'n_positives_mean',
    'pull_loss', 'push_loss', 'mean_pos_dist', 'mean_neg_dist', 'dedup_fraction',
    'positives_per_anchor', 'negatives_per_anchor',
    'positive_fallback_rate', 'mean_hard_negatives', 'temperature',
    'positive_fallback_rate', 'mean_hard_negatives', 'temperature',
    'z_pred_std', 'z_tgt_std', 'z_std', 'c_std', 'p_std', 'cos_sim_offdiag',
    'plateau', 'excess', 'emb_std',
):
    value = tjm.output_metric(out, key)
    try:
        value = float(value.detach().cpu())
    except AttributeError:
        value = float(value)
    if not math.isnan(value):
        print(f'{key}:', value)

objective_class: jepa
resolved_objective_class: jepa
view_mode: hard_disjoint_future
predictor: mlp_multi_pos
horizon: 4
loss: 0.5573079586029053
smooth_l1_loss: 0.5573079586029053
z_pred_std: 0.4810827374458313
z_tgt_std: 0.852410614490509
z_std: 0.5403733849525452
c_std: 0.5756270289421082
p_std: 0.2944149374961853
cos_sim_offdiag: 0.5026025176048279


## 7. Smoke training run

In [11]:

import sys
import importlib

if 'require_v8_pair_index_ready' in globals():
    require_v8_pair_index_ready()
elif OBJECTIVE_CLASS in {'jepa_order_aware', 'order_aware', 'v8'}:
    raise RuntimeError('Run the v8 offline pair-index cell before training.')

SMOKE_OUT = Path('/content/_jepa_smoke')
sys.argv = [
    'train_jepa.py',
    '--config', str(CONFIG_PATH),

    # Runtime/path overrides for the smoke test.
    '--data_dir', str(DATA_DIR),
    '--li_split_train_chunks', '1',
    '--out_dir', str(SMOKE_OUT),
    '--drive_sync_dir', '',

    # Tiny model overrides so this cell is a fast integration check.
    '--n_layers', '2',
    '--n_heads', '4',
    '--d_model', '64',
    '--predictor_hidden_dim', '128',
    '--predictor_n_layers', '2',
    '--predictor_n_heads', '4',
    '--batch_size', '256',
    '--warmup_steps', '50',
    '--passes_over_data', '1',
    '--eval_chunks', '1',
    '--eval_every_chunks', '1',
    '--max_chunks', '1',
] + PAIR_INDEX_RUNTIME_ARGS

import othello_research.objectives.jepa_hard_disjoint_action as jhda
import othello_research.objectives.jepa_order_aware as joa
import othello_research.objectives.jepa_hard_disjoint_infonce as jhdi
import othello_research.objectives.jepa as jepa_obj
import othello_research.datasets.order_aware_pair_index as order_pair_index
import othello_research.datasets.order_aware_dataset as order_dataset
import othello_research.training.train_jepa_order_aware as order_training
import othello_research.training.train_jepa as tjm
importlib.reload(jhda)
importlib.reload(joa)
importlib.reload(jhdi)
importlib.reload(jepa_obj)
importlib.reload(order_pair_index)
importlib.reload(order_dataset)
importlib.reload(order_training)
importlib.reload(tjm)
tjm.main()

estimated total_steps=391 (manifest, ~99,997 games over 1 pass(es))
device: cuda
train chunks: 1
val chunks: 237
vocab_size: 61, block_size: 59
jepa: objective_class=jepa variant=v4 view_mode=hard_disjoint_future loss=smooth_l1 predictor=mlp_multi_pos horizon=4 position_sampling=all ema=0.996 precision=bf16
milestone games: [1000000, 5000000, 10000000, 15000000, 20000000]
fraction checkpoints: []
chunk 1/1 starting: gen10e5__20220324_153929.pickle (pass 1, order 1/1)
jepa pass=1 chunk=1/1 step=391 games=99,997 samples=5,497,446 smooth_l1=0.1607 pred_std=0.104 tgt_std=0.412 z_std=0.412 c_std=0.245 p_std=0.103 cos=0.784 pos/game=55.0 loss=0.1607 sample_skip=0.00% | val: loss=0.0197 smooth_l1=0.0197 pred_std=0.017 tgt_std=0.153 z_std=0.153 c_std=0.032 p_std=0.014 cos=0.974 pos/game=55.0 sample_skip=0.00% dt=22s


In [12]:
import sys
import importlib
from pathlib import Path

PILOT_OUT = Path('/content/runs/jepa_v4a_disjoint_future_pilot_5chunks')

sys.argv = [
    'train_jepa.py',
    '--config', str(CONFIG_PATH),

    # Runtime/path overrides
    '--data_dir', str(DATA_DIR),
    '--out_dir', str(PILOT_OUT),
    '--drive_sync_dir', '',

    # Pilot size
    '--li_split_train_chunks', '5',
    '--max_chunks', '5',
    '--passes_over_data', '1',

    # Keep validation cheap but informative
    '--eval_chunks', '1',
    '--eval_every_chunks', '1',

    # Keep v4 config architecture/objective unchanged
    # Do NOT override n_layers, n_heads, d_model, predictor, etc.

    # Optional: keep batch same as config unless memory fails
    # '--batch_size', '512',
]

import othello_research.objectives.jepa_hard_disjoint_action as jhda
import othello_research.objectives.jepa_order_aware as joa
import othello_research.objectives.jepa_hard_disjoint_infonce as jhdi
import othello_research.objectives.jepa as jepa_obj
import othello_research.datasets.order_aware_pair_index as order_pair_index
import othello_research.datasets.order_aware_dataset as order_dataset
import othello_research.training.train_jepa_order_aware as order_training
import othello_research.training.train_jepa as tjm
importlib.reload(jhda)
importlib.reload(joa)
importlib.reload(jhdi)
importlib.reload(jepa_obj)
importlib.reload(order_pair_index)
importlib.reload(order_dataset)
importlib.reload(order_training)
importlib.reload(tjm)
tjm.main()

estimated total_steps=1954 (manifest, ~499,985 games over 1 pass(es))
device: cuda
train chunks: 5
val chunks: 233
vocab_size: 61, block_size: 59
jepa: objective_class=jepa variant=v4 view_mode=hard_disjoint_future loss=smooth_l1 predictor=mlp_multi_pos horizon=4 position_sampling=all ema=0.996 precision=bf16
milestone games: [1000000, 5000000, 10000000, 15000000, 20000000]
fraction checkpoints: []
chunk 1/5 starting: gen10e5__20220324_153933.pickle (pass 1, order 1/5)
jepa pass=1 chunk=1/5 step=391 games=99,999 samples=5,497,565 smooth_l1=0.1787 pred_std=0.101 tgt_std=0.469 z_std=0.469 c_std=0.219 p_std=0.101 cos=0.758 pos/game=55.0 loss=0.1787 sample_skip=0.00% | val: loss=0.0838 smooth_l1=0.0838 pred_std=0.086 tgt_std=0.403 z_std=0.403 c_std=0.146 p_std=0.085 cos=0.822 pos/game=55.0 sample_skip=0.00% dt=50s
chunk 2/5 starting: gen10e5__20220324_154028.pickle (pass 1, order 2/5)
jepa pass=1 chunk=2/5 step=782 games=199,998 samples=10,995,716 smooth_l1=0.1191 pred_std=0.133 tgt_std=0.

## 8. Full training run

In [13]:
import sys
import importlib
import shutil
import traceback
from pathlib import Path
from google.colab import runtime

should_disconnect = True

try:
    if 'require_v8_pair_index_ready' in globals():
        require_v8_pair_index_ready()
    elif OBJECTIVE_CLASS in {'jepa_order_aware', 'order_aware', 'v8'}:
        raise RuntimeError('Run the v8 offline pair-index cell before training.')
    LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

    RESUME_CKPT = LOCAL_OUT_DIR / 'latest.pt'
    DRIVE_CKPT = DRIVE_OUT_DIR / 'latest.pt'

    if not RESUME_CKPT.exists() and DRIVE_CKPT.exists():
        shutil.copy2(DRIVE_CKPT, RESUME_CKPT)
        print(f'Copied resume checkpoint from Drive: {DRIVE_CKPT} -> {RESUME_CKPT}')

    resume_args = []
    if RESUME_CKPT.exists():
        resume_args = ['--resume', str(RESUME_CKPT)]
        print('Resuming from:', RESUME_CKPT)
    else:
        print('No checkpoint found. Starting fresh run.')

    sys.argv = [
        'train_jepa.py',
        '--config', str(CONFIG_PATH),

        # Notebook-specific runtime path overrides. Experiment settings stay in YAML.
        '--data_dir', str(DATA_DIR),
        '--out_dir', str(LOCAL_OUT_DIR),
        '--drive_sync_dir', str(DRIVE_OUT_DIR),
    ] + PAIR_INDEX_RUNTIME_ARGS + resume_args

    import othello_research.objectives.jepa as jepa_obj
    import othello_research.datasets.order_aware_pair_index as order_pair_index
    import othello_research.datasets.order_aware_dataset as order_dataset
    import othello_research.training.train_jepa_order_aware as order_training
    import othello_research.training.train_jepa as tjm
    importlib.reload(jepa_obj)
    importlib.reload(order_pair_index)
    importlib.reload(order_dataset)
    importlib.reload(order_training)
    importlib.reload(tjm)
    tjm.main()

except Exception as e:
    print('\nTraining stopped with an exception:')
    print(type(e).__name__, e)
    traceback.print_exc()

finally:
    print('\nFinal safety sync to Drive before disconnect...')
    try:
        import othello_research.datasets.order_aware_pair_index as order_pair_index
        import othello_research.datasets.order_aware_dataset as order_dataset
        import othello_research.training.train_jepa_order_aware as order_training
        import othello_research.training.train_jepa as tjm
        n_synced, sync_dt = tjm.sync_to_drive(Path(LOCAL_OUT_DIR), Path(DRIVE_OUT_DIR))
        print(f'Final sync completed: {n_synced} file(s) in {sync_dt:.1f}s')
    except Exception as sync_error:
        print('Final sync failed, but runtime will still disconnect.')
        print(type(sync_error).__name__, sync_error)

    if should_disconnect:
        print('Disconnecting Colab runtime now...')
        runtime.unassign()

No checkpoint found. Starting fresh run.
estimated total_steps=78125 (manifest, ~19,999,840 games over 1 pass(es))
device: cuda
train chunks: 200
val chunks: 38
vocab_size: 61, block_size: 59
jepa: objective_class=jepa variant=v4 view_mode=hard_disjoint_future loss=smooth_l1 predictor=mlp_multi_pos horizon=4 position_sampling=all ema=0.996 precision=bf16
milestone games: [1000000, 5000000, 10000000, 15000000, 20000000]
fraction checkpoints: []
drive auto-sync: every 5 chunk(s) -> /content/drive/MyDrive/Master_Thesis_Artifacts/runs/jepa_v4_multi_pos_k4_hd_run_001_allpos_bf16
chunk 1/200 starting: gen10e5__20220324_154407.pickle (pass 1, order 1/200)
jepa pass=1 chunk=1/200 step=391 games=99,994 samples=5,497,130 smooth_l1=0.1784 pred_std=0.101 tgt_std=0.468 z_std=0.468 c_std=0.218 p_std=0.101 cos=0.759 pos/game=55.0 loss=0.1784 sample_skip=0.00% dt=29s
chunk 2/200 starting: gen10e5__20220324_161410.pickle (pass 1, order 2/200)
jepa pass=1 chunk=2/200 step=782 games=199,994 samples=10,99

## 9. Resume after disconnect

In [ ]:
import sys
import importlib
import shutil

RESUME_CKPT = LOCAL_OUT_DIR / 'latest.pt'
DRIVE_CKPT = DRIVE_OUT_DIR / 'latest.pt'
if not RESUME_CKPT.exists() and DRIVE_CKPT.exists():
    LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_CKPT, RESUME_CKPT)
assert RESUME_CKPT.exists(), RESUME_CKPT

if 'require_v8_pair_index_ready' in globals():
    require_v8_pair_index_ready()
elif OBJECTIVE_CLASS in {'jepa_order_aware', 'order_aware', 'v8'}:
    raise RuntimeError('Run the v8 offline pair-index cell before training.')

sys.argv = [
    'train_jepa.py',
    '--config', str(CONFIG_PATH),

    # Notebook-specific runtime path overrides. Experiment settings stay in YAML.
    '--data_dir', str(DATA_DIR),
    '--out_dir', str(LOCAL_OUT_DIR),
    '--drive_sync_dir', str(DRIVE_OUT_DIR),
    '--resume', str(RESUME_CKPT),
] + PAIR_INDEX_RUNTIME_ARGS

import othello_research.objectives.jepa as jepa_obj
import othello_research.datasets.order_aware_pair_index as order_pair_index
import othello_research.datasets.order_aware_dataset as order_dataset
import othello_research.training.train_jepa_order_aware as order_training
import othello_research.training.train_jepa as tjm
importlib.reload(jepa_obj)
importlib.reload(order_pair_index)
importlib.reload(order_dataset)
importlib.reload(order_training)
importlib.reload(tjm)
tjm.main()


## 10. Drive sync

In [ ]:
from pathlib import Path
import shutil
import yaml

# CONFIG_NAME is the single source of truth for every experiment-specific path.
# After a runtime restart, re-run Section 3 (Configure paths) or set CONFIG_NAME
# explicitly before using this manual sync cell.
if 'LOCAL_OUT_DIR' not in globals() or 'DRIVE_OUT_DIR' not in globals():
    if 'CONFIG_NAME' not in globals():
        raise NameError(
            'CONFIG_NAME is not defined. Re-run Section 3 (Configure paths) '
            'before syncing to Drive.'
        )
    PROJECT_ROOT = globals().get(
        'PROJECT_ROOT',
        Path('/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code'),
    )
    ARTIFACT_ROOT = globals().get(
        'ARTIFACT_ROOT',
        Path('/content/drive/MyDrive/Master_Thesis_Artifacts'),
    )
    RUNS_DIR = globals().get('RUNS_DIR', ARTIFACT_ROOT / 'runs')
    CONFIGS_DIR = globals().get('CONFIGS_DIR', PROJECT_ROOT / 'configs')
    CONFIG_PATH = CONFIGS_DIR / CONFIG_NAME
    assert CONFIG_PATH.exists(), CONFIG_PATH
    with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
        EXPERIMENT_CONFIG = yaml.safe_load(f)
    RUN_NAME = EXPERIMENT_CONFIG['experiment']['name']
    LOCAL_OUT_DIR = Path('/content/runs') / RUN_NAME
    DRIVE_OUT_DIR = RUNS_DIR / RUN_NAME

if not LOCAL_OUT_DIR.exists():
    raise FileNotFoundError(
        f'LOCAL_OUT_DIR does not exist: {LOCAL_OUT_DIR}. Run a training cell first.'
    )

DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
n_copied = 0
for src in sorted(LOCAL_OUT_DIR.iterdir()):
    if not src.is_file():
        continue
    dst = DRIVE_OUT_DIR / src.name
    if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
        shutil.copy2(src, dst)
        n_copied += 1
print('copied files:', n_copied)
print('local dir:', LOCAL_OUT_DIR)
print('drive dir:', DRIVE_OUT_DIR)

## 11. Quick metrics view

In [ ]:
metrics_path = LOCAL_OUT_DIR / 'metrics.csv'
if not metrics_path.exists():
    metrics_path = DRIVE_OUT_DIR / 'metrics.csv'
import csv
cols = ['time', 'chunk_count', 'step', 'train_loss', 'smooth_l1_loss', 'z_pred_std', 'z_tgt_std', 'train_sample_skip_ratio', 'train_batch_skip_ratio', 'val_loss', 'val_sample_skip_ratio', 'lr']
with open(metrics_path, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
for row in rows[-5:]:
    print({c: row.get(c, '') for c in cols if c in row})


## 12. Notes on different runs

## Running different configurations

The notebook is now config-driven. To launch a different JEPA experiment, change only `CONFIG_NAME` in Section 3.

The YAML controls:
- target topology: `experiment.variant` (`jepa_v1` shared/single encoder or `jepa_v2` EMA target encoder)
- predictor architecture: `predictor_type`, `predictor_hidden_dim`, `predictor_n_layers`, `predictor_dropout`, `predictor_activation`
- objective: `loss_type` (`mse`, `smooth_l1`, or `vicreg`), `prediction_horizon`, `ema_momentum`, VICReg weights, Smooth L1 beta
- optimization: batch size, LR, warmup, precision, pass count

The notebook only overrides runtime paths (`data_dir`, local `out_dir`, Drive sync dir) and resume checkpoint handling, so Colab can use the same experiment config without hardcoding paths into the trainer.
